In [ ]:
import sys
sys.path.append("..")
import torch

from src.environment import MultiCurrencyEnv
from src.simulator import geometric_brownian_step

from bokeh.palettes import Category10
from bokeh.models import HoverTool, ColumnDataSource
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
N = 4
C0 = 10_000.0
w0 = torch.zeros(N)
p0 = torch.tensor([100., 50., 25., 10.])
sell_fee = torch.full((N,), 0.001)
buy_fee  = torch.full((N,), 0.001)

taus_price = torch.tensor([1., 5., 10., 20., 50., 100.])
vol_taus   = torch.tensor([10., 50., 100.])
volu_taus  = torch.tensor([10., 50., 100.])

env = MultiCurrencyEnv(C0, w0, p0, sell_fee, buy_fee,
                       taus_price, dt=1.0,
                       vol_taus=vol_taus, volu_taus=volu_taus,
                       reward_mode="log",transaction_eps=1e-4)

a_hist = []
V_hist = []
C_hist = []
w_hist = []
prices_hist = []
volume_hist = []

p_rel_hist = []
x_hist = []
sigma_hist = []
v_rel_hist = []
rew_hist = []

T = 100000

state = env.reset()
for t in range(T):
    a = torch.tanh(torch.randn(N+1))

    new_prices = geometric_brownian_step(env.p, mu=0.001, sigma=0.05)
    new_volume = torch.rand(N) * 1000
    env.update_prices(new_prices)
    env.update_volume(new_volume)
    state, reward, done, info = env.step(a)

    prices_hist.append(new_prices)
    volume_hist.append(new_volume)

    a_hist.append(info['a'])
    V_hist.append(info['V'])
    C_hist.append(info['C'])
    w_hist.append(info['w'])

    p_rel_hist.append(state.p_rel.detach().cpu().numpy())
    x_hist.append(state.x.detach().cpu().numpy())
    sigma_hist.append(state.sigma.detach().cpu().numpy())
    v_rel_hist.append(state.v_rel.detach().cpu().numpy())
    rew_hist.append(reward)


In [ ]:
t = list(range(T))

In [ ]:
f1 = bk.figure(title=f"Prices", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    pi = [prices_hist[k][i] for k in range(T)]
    r = f1.line(t, pi, line_width=2, legend_label=f"Asset {i+1}", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Prices", x_axis_label="t", y_axis_label="USD", width=900, height=320)

for i in range(N):
    vi = [volume_hist[k][i] * prices_hist[k][i] for k in range(T)]
    r = f1.line(t, vi, line_width=2, legend_label=f"Asset {i+1}", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Portfolio Value", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
r1 = f1.line(t, V_hist, line_width=2, legend_label="Total V")
r2 = f1.line(t, C_hist, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
f1.add_tools(HoverTool(renderers=[r1], tooltips=[("t", "@t"), ("V", "@V{0,0.00}")], mode="vline"))
bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Rewards", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

r = f1.line(t, rew_hist, line_width=2, legend_label=f"Reward")

ret_hist = []
gamma = 0.99
G = 0.0
for r in reversed(rew_hist):
    G = r + gamma * G
    ret_hist.insert(0, G)

r = f1.line(t, ret_hist, line_width=2, legend_label=f"Return", color="green")
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Actions", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

a0 = [a_hist[k][0] for k in range(T)]
f1.scatter(t, a0, legend_label=f"Investment fraction", color=Category10[10][0])
for i in range(1,N+1):
    ai = [a_hist[k][i] for k in range(T)]
    f1.scatter(t, ai, legend_label=f"Asset {i} fraction to buy/sell", color=Category10[10][i])

f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Asset Position", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

x0 = [x_hist[k][0] for k in range(T)]
f1.line(t, x0, line_width=2, legend_label=f"Cash fraction", color=Category10[10][0])
for i in range(1,N+1):
    xi = [x_hist[k][i] for k in range(T)]
    f1.line(t, xi, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative price change (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.taus_p)):
        p_ij = [p_rel_hist[k][j,i] for k in range(T)]
        f1.line(t, p_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.taus_p))
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative Volatility (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.vol_taus)):
        sigma_ij = [sigma_hist[k][j,i] for k in range(T)]
        f1.line(t, sigma_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.vol_taus))
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Relative Volume (EWMA)", x_axis_label="t", y_axis_label="Fraction of Portfolio Value", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i in range(N):
    for j in range(len(env.volu_taus)):
        v_rel_ij = [v_rel_hist[k][j,i] for k in range(T)]
        f1.line(t, v_rel_ij, line_width=2, legend_label=f"Asset {i} fraction", color=Category10[10][i], alpha=0.2 + 0.8 * j / len(env.volu_taus))
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
from src.network import QNetwork, SquashedStochasticVanillaNetwork
from src.agent import EntropyRegAgent

agent = EntropyRegAgent(
    state_size=env.state_size,
    action_size=env.action_size,
    actor=SquashedStochasticVanillaNetwork(
        env.state_size, env.action_size,
        sizes=[128, 128], device="cpu"
    ),
    critic=QNetwork(
        env.state_size, env.action_size,
        sizes=[128, 128], device="cpu"
    ),
    actor_lr=1e-4,
    critic_lr=1e-4,
    gamma=0.99,
    tau=100.0,
    alpha=0.01,
    device="cpu"
)

In [ ]:
def train_on_historical(agent, hist_p, hist_v, env,
        n_episodes,
        batch_size=32,
        update_interval=100,
        n_updates=8,
        max_steps=5000,
        store=1,
        actor_lr=1e-4,
        critic_lr=1e-4,
        actor_m=0.0,
        critic_m=0.0):
    """Train the agent in the given environment."""
    assert len(hist_p) == len(hist_v), "Price and volume history must have the same length."
    
    agent.update_optimizers(actor_lr, critic_lr, actor_m, critic_m)

    if store:
        total_loss_a = []
        total_loss_c = []
        total_loss_alpha = []
        total_reward = []
        total_info = []

    for episode in range(1, n_episodes+1):
        start = torch.randint(len(hist_p)-max_steps-1, size=[1]).item()
        state = env.reset(
            C0=env._defaults['C0'],
            w0=torch.zeros(env.N),
            p0=hist_p[start],
        ).to_tensor()

        if store:
            episode_loss_a = []
            episode_loss_c = []
            episode_loss_alpha = []
            episode_reward = []
            episode_info = []

        counter = 1
        for i in range(max_steps):
            action = agent.act(state, deterministic=False)

            env.update_prices(hist_p[start+i])
            env.update_volume(hist_v[start+i])
            
            next_state, reward, done, info = env.step(action)
            next_state = next_state.to_tensor()

            agent.buffer.store(
                state.detach(),
                action.detach(),
                next_state.detach(),
                reward,
                done
            )
            episode_reward.append(reward)
            episode_info.append(info)
        
            if counter % update_interval == 0:
                for _ in range(n_updates):
                    loss_a, loss_c, loss_alpha = agent.train_step(batch_size)
                    episode_loss_a.append(loss_a)
                    episode_loss_c.append(loss_c)
                    episode_loss_alpha.append(loss_alpha)
                print(f"episode {episode} - reward: {sum(episode_reward):.2f}", end="\r")

            if done or counter > max_steps:
                break

            state = next_state
            counter += 1

        if store:
            total_loss_a.append(torch.tensor(episode_loss_a))
            total_loss_c.append(torch.tensor(episode_loss_c))
            total_loss_alpha.append(torch.tensor(episode_loss_alpha))
            total_reward.append(torch.tensor(episode_reward))
            total_info.append(episode_info)

        print(f"episode {episode} - loss_a: {sum(episode_loss_a)/counter:.5f} - loss_c: {sum(episode_loss_c)/counter:.5f} - loss_alpha: {sum(episode_loss_alpha)/counter:.5f} - reward: {sum(episode_reward):.5f}")
    
    if store:
        return (
            torch.stack(total_loss_a),
            torch.stack(total_loss_c),
            torch.stack(total_loss_alpha),
            torch.stack(total_reward),
            total_info
        )

In [ ]:
loss_a, loss_c, loss_alpha, rewards, info = train_on_historical(agent, hist_p=prices_hist, hist_v=volume_hist, env=env, n_episodes=10, batch_size=32)

In [ ]:
fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
fig.line(torch.arange(loss_a.numel()) / loss_a.shape[1], loss_a.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][0], alpha=0.3)
fig.line(torch.arange(len(loss_a)) + 0.5, loss_a.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][0])
bk.show(fig)

In [ ]:
fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
fig.line(torch.arange(loss_c.numel()) / loss_c.shape[1], loss_c.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][2], alpha=0.3)
fig.line(torch.arange(len(loss_c)) + 0.5, loss_c.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][2])
bk.show(fig)

In [ ]:
fig = bk.figure(title="Alpha Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
fig.line(torch.arange(loss_alpha.numel()) / loss_alpha.shape[1], loss_alpha.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][3], alpha=0.3)
fig.line(torch.arange(len(loss_alpha)) + 0.5, loss_alpha.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][3])
bk.show(fig)

In [ ]:
fig = bk.figure(title="Rewards", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][4], alpha=0.3)
fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][4])
bk.show(fig)

In [ ]:
def get_prices(info):
    prices = []
    for episode_info in info:
        prices.append([step['p'] for step in episode_info])
    return prices

In [ ]:
prices = get_prices(info)

for p_hist in prices:
    f1 = bk.figure(title=f"Prices", x_axis_label="t", y_axis_label="USD", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

    T = len(p_hist)
    for i in range(N):
        pi = [p_hist[k][i] for k in range(T)]
        r = f1.line(torch.arange(T), pi, line_width=2, legend_label=f"Asset {i+1}", color=Category10[10][i])
    f1.legend.click_policy = "hide"

    bk.show(f1)